# Tally — Kaggle notebook

This is the Kaggle port of the Lexsi Tally working notebook (fine-tuning Tiny Aya to extract fields from forest carbon project documents).

**After any session restart, do this:**
1. Confirm the accelerator is a GPU (Settings → Accelerator → GPU T4 x2).
2. Confirm the `tally-pdds` dataset is attached as an input, Add-ons → Secrets has `HF_TOKEN`, and Settings → Internet is on (needed to install packages and download the model).
3. Run the **Setup** cell below. It installs the missing packages, logs in to Hugging Face, and loads Tiny Aya onto the first T4 (`cuda:0`). The model is downloaded to `/kaggle/tmp/hf`, which is local scratch and is wiped when the session ends. We do not keep a Hugging Face cache on `/kaggle/working`, because the shortcut (symlink) layout that cache uses broke when we tried the same thing on Google Drive.
4. Then run only the later cells whose work isn't already saved under `/kaggle/working/tally`. Helpers skip work when the output file already exists.

`/kaggle/working` is the 20 GB folder Kaggle keeps between sessions. Your extracted text, labels, evals, and LoRA checkpoints live there. The ~7 GB model does not, so a new session spends a couple of minutes re-downloading it.

The original Colab notebooks are left unchanged.


## 1. Setup (run this one after any restart)

This one cell does everything needed to get back to a ready model:

- installs CuratorKIT, AlignTune, AuditKIT, and PyMuPDF. Kaggle already has torch and a recent transformers, so we don't reinstall those unless the version is older than 5 (we need transformers 5 for `dtype=` instead of the older `torch_dtype=`, and for `apply_chat_template` returning a BatchEncoding)
- logs in with your `HF_TOKEN` Kaggle secret (Add-ons → Secrets). The token is never printed.
- loads Tiny Aya in float16 on the first T4 (`cuda:0`), unless it is already in memory

This machine has two T4s, 16 GB of video memory each. Tiny Aya in float16 is about 7 GB, so it fits on one card with room for the KV cache (the stored attention keys and values for the tokens the model is reading). We pin this copy to `cuda:0`. A second copy can go on `cuda:1` later, for parallel extraction across documents.

The tokenizer and weights are cached under `/kaggle/tmp/hf`. That's session scratch, not the 20 GB persistent `/kaggle/working` disk. We learned the hard way on Drive that Hugging Face's cache uses shortcuts (symlinks) the host filesystem can turn into tiny text files, after which the loader fails with errors like "expected value at line 1 column 1" or "header too large". Re-downloading a few minutes each session is safer.


In [ ]:
# === 1. SETUP: install packages, log in, and load Tiny Aya ===
# Run this after any session restart. Loads the model onto the first T4.

import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Kaggle already has torch. Only install what's missing.
# transformers>=5 is a no-op if the image is already new enough; we need it
# for dtype= (not the older torch_dtype=) and for apply_chat_template
# returning a BatchEncoding.
!pip install -q "curatorkit[all]" aligntune "auditkit[transformers]" pymupdf "transformers>=5"

import torch
import curatorkit, aligntune, auditkit
from transformers import AutoTokenizer, AutoModelForCausalLM

# Never print the token.
login(UserSecretsClient().get_secret("HF_TOKEN"))

MID = "CohereLabs/tiny-aya-global"
HF_CACHE = "/kaggle/tmp/hf"
os.makedirs(HF_CACHE, exist_ok=True)

# Re-running this cell after a restart is cheap if the model is still in RAM.
# A full session reset clears RAM and /kaggle/tmp, so the model will download again.
if "model" not in globals():
    tok = AutoTokenizer.from_pretrained(MID, cache_dir=HF_CACHE)
    model = AutoModelForCausalLM.from_pretrained(
        MID,
        dtype=torch.float16,  # not the older torch_dtype=
        device_map={"": 0},   # pin this copy to the first T4
        cache_dir=HF_CACHE,
    )

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
print("ready")


## 2. Project folders

All Tally outputs live under `/kaggle/working/tally`. `/kaggle/working` is the 20 GB folder Kaggle keeps between sessions. Subfolders:

- `text` — extracted text from the project PDFs
- `labels` — structured JSON we'll create later
- `evals` — scoring results from AuditKIT
- `checkpoints` — LoRA weights saved during fine-tuning

The PDFs themselves come from a Kaggle dataset attached as input, under `/kaggle/input/...`. We don't copy them into `/kaggle/working`.

`exist_ok=True` means this cell is safe to re-run; it won't fail if the folders are already there.


In [ ]:
# === 2. FOLDERS: create the persistent Tally output directories ===
# Makes text, labels, evals, checkpoints, and results under /kaggle/working/tally.

import os

TALLY = "/kaggle/working/tally"

for name in ("text", "labels", "evals", "checkpoints", "results"):
    os.makedirs(os.path.join(TALLY, name), exist_ok=True)

print("TALLY =", TALLY)
print("folders:", ", ".join(sorted(os.listdir(TALLY))))


## 3. Helpers

Small tools we'll reuse in later cells:

- `cached_json(path, fn)` and `cached_text(path, fn)` — if the file already exists, read and return it. If not, call `fn()`, save the result, and return it. After a restart you don't redo finished work.
- `ask(prompt, max_new_tokens=900, temperature=None)` — send a chat message to Tiny Aya and return only the reply text. The default is 900 so a 7-field JSON reply is not cut off mid-quote (200 was too small). Greedy decoding is the default (same answer each time, which is what we want for extraction). Pass a `temperature` to sample instead.
- `parse_json(text)` — pull the first `{...}` block out of a model reply, then parse it. Chat-tuned models often wrap JSON in a markdown code fence such as `` ```json ... ``` ``. The raw string still has the right content, but `json.loads` rejects the fences. This parser is what keeps Tally's "validated JSON" step safe.

In current Transformers, `apply_chat_template(..., return_tensors="pt")` returns a `BatchEncoding` (a dict-like object with `.input_ids` and `.attention_mask`), not a bare tensor. `ask` uses those fields so generation works.


In [ ]:
# === 3. HELPERS: cached file I/O, JSON parser, and the ask() call ===
# Reusable tools for later cells. ask() defaults to 900 new tokens so JSON replies are not cut off.

import json
import re
from pathlib import Path


def cached_json(path, fn):
    path = Path(path)
    if path.exists():
        with path.open(encoding="utf-8") as f:
            return json.load(f)
    data = fn()
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    return data


def cached_text(path, fn):
    path = Path(path)
    if path.exists():
        return path.read_text(encoding="utf-8")
    text = fn()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return text


def parse_json(text):
    # Pull the first JSON object out of a model reply and parse it.
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise json.JSONDecodeError("no JSON object found", text, 0)
    return json.loads(match.group(0))


def ask(prompt, max_new_tokens=900, temperature=None):
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    gen_kwargs = {
        "input_ids": inputs.input_ids,
        "attention_mask": inputs.attention_mask,
        "max_new_tokens": max_new_tokens,
    }
    if temperature is None:
        gen_kwargs["do_sample"] = False  # greedy: pick the highest-scoring token
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature

    output = model.generate(**gen_kwargs)
    # Slice off the prompt tokens so we only decode the model's reply.
    reply_ids = output[0][inputs.input_ids.shape[1]:]
    return tok.decode(reply_ids, skip_special_tokens=True)


## 4. Smoke test

Ask the model for a tiny JSON object, print the raw reply with `repr` so any markdown fences are visible, then parse it with `parse_json`. This is a quick check that (a) generation works and (b) the base model can already produce structured output — the skill Tally will train it to do well on project documents. Seeing the fences in the raw reply is why the parser exists.


In [ ]:
# === 4. SMOKE TEST: check that Tiny Aya can reply with JSON ===
# Sends a tiny prompt and parses the reply, so we know generation and parse_json work.

import json

reply = ask('Reply with JSON: {"status": "ok"}', max_new_tokens=30)
print("Model reply:")
print(repr(reply))
print()

try:
    parsed = parse_json(reply)
    print("Parsed OK:", parsed)
except json.JSONDecodeError as err:
    print("Did not parse as JSON:", err)


## 5. Anatomy

A first look inside Tiny Aya:

- `print(model)` shows the layer tree: the embedding table, 36 decoder layers (each with attention and an MLP), then the output head. The embeddings and the output head share the same weights (tied embeddings), so you won't see a separate big matrix for `lm_head`.
- `model.config` shows the numbers behind it (layer count, hidden size, attention heads, vocabulary size, and the 8,192-token context window).
- The parameter count and the first few weight shapes show the actual matrices. In float16 this is about 7 GB of video memory, which is why one T4 (16 GB) is enough, and why a second copy can fit on the other T4 later.


In [ ]:
# === 5. ANATOMY: print Tiny Aya's layers, config, and parameter count ===
# A first look inside the model: 36 decoder layers, about 7 GB in float16, 8,192-token context.

print(model)
print(model.config)
total = sum(p.numel() for p in model.parameters())
print(f"parameters: {total/1e9:.2f} B")
for name, p in list(model.named_parameters())[:12]:
    print(name, tuple(p.shape))


## 6. PDF to text, cached

Read every PDF under `/kaggle/input` (however the dataset is slugged), extract the text of each page, and save it to `TALLY/text/<name>.txt`. If that `.txt` already exists, we skip the extraction (the `cached_text` helper).

For each file we also print:

- the page count
- an approximate token count, using Tiny Aya's own tokenizer on the full text

**Important:** Tiny Aya can only attend to **8,192 tokens** at a time (`max_position_embeddings`). Many of these project documents are far longer than that, so later we'll split each document into sections and extract fields from the relevant ones, rather than stuffing a whole PDD into one prompt.


In [ ]:
# === 6. PDF TO TEXT: extract every attached PDF into TALLY/text ===
# Skips files that already have a .txt. Prints page and token counts, and flags docs over 8,192 tokens.

try:
    import pymupdf as fitz
except ImportError:
    import fitz
from pathlib import Path

text_dir = Path(TALLY) / "text"
text_dir.mkdir(parents=True, exist_ok=True)

pdfs = sorted(Path("/kaggle/input").glob("**/*.pdf"))
print(f"Found {len(pdfs)} PDFs under /kaggle/input")

for pdf in pdfs:
    out_path = text_dir / f"{pdf.stem}.txt"

    def extract(pdf_path=pdf):
        doc = fitz.open(pdf_path)
        pages = []
        for i, page in enumerate(doc, start=1):
            pages.append(f"--- page {i} ---\n{page.get_text()}")
        doc.close()
        return "\n\n".join(pages)

    text = cached_text(out_path, extract)
    page_count = text.count("--- page ")
    n_tokens = len(tok.encode(text))
    over = "  (over the 8,192-token window)" if n_tokens > 8192 else ""
    print(f"{pdf.name}: {page_count} pages, ~{n_tokens:,} tokens{over} -> {out_path.name}")


## 7. Chunk documents and extract notes

Tiny Aya can only read **8,192 tokens** at a time, and most of these project documents are much longer than that. The cells below (7a–7h) are short on purpose, so they paste cleanly into the Kaggle editor.

They split each extracted file into overlapping chunks of at most 6,000 tokens, ask the model for the 7 Tally fields from every chunk, recover the page number from the quote in Python, and merge those notes into one JSON log per document. Results are written to `TALLY/results/<stem>.json` after each file, so a crash does not lose finished work.

A full pass is 77 chunks. On one T4, plan on about 15–30 seconds per chunk, or roughly 20–40 minutes in total (budget up to an hour if some replies retry). The model already fits on `cuda:0`; splitting it with `device_map="auto"` would not make this faster.

Import this notebook (File → Import notebook) or paste these cells in the Kaggle UI. Do not push this notebook with the Kaggle API: that dropped the `HF_TOKEN` secret last time.


In [ ]:
# === 7a. PAGE SPLITTER: cut extracted text at --- page N --- markers ===
# Returns (page_number, page_text) pairs so later chunks still know which page a quote came from.

import re


def parse_pages(text):
    marker = re.compile(r"--- page (\d+) ---")
    matches = list(marker.finditer(text))
    if not matches:
        return [(1, text)]
    pages = []
    for i, m in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        pages.append((int(m.group(1)), text[m.start():end].strip("\n")))
    return pages


In [ ]:
# === 7b. CHUNKER: splits a document into 3,500-token pieces ===
# Overlapping windows (300-token overlap) so a fact on a boundary is less likely to be split in half.

def chunk_doc(text, tokenizer, max_tokens=3500, overlap=300):
    token_ids, page_of = [], []
    for page_num, page_text in parse_pages(text):
        piece = page_text if not token_ids else "\n\n" + page_text
        ids = tokenizer.encode(piece, add_special_tokens=False)
        token_ids.extend(ids)
        page_of.extend([page_num] * len(ids))
    if not token_ids:
        return []
    chunks, start, n = [], 0, len(token_ids)
    while start < n:
        end = min(start + max_tokens, n)
        chunk_text = tokenizer.decode(token_ids[start:end])
        if not chunk_text.lstrip().startswith("--- page "):
            chunk_text = f"--- page {page_of[start]} ---\n" + chunk_text
        chunks.append({
            "text": chunk_text,
            "start_page": page_of[start],
            "end_page": page_of[end - 1],
            "n_tokens": len(tokenizer.encode(chunk_text, add_special_tokens=False)),
        })
        if end >= n:
            break
        nxt = end - overlap
        start = nxt if nxt > start else end
    return chunks


In [ ]:
# === 7c. CHUNK ALL FILES: run the chunker on every .txt under TALLY/text ===
# Stores results in all_chunks and prints the chunk count for each document.

from pathlib import Path

all_chunks = {}
for path in sorted((Path(TALLY) / "text").glob("*.txt")):
    chunks = chunk_doc(path.read_text(encoding="utf-8"), tok)
    all_chunks[path.name] = chunks
    print(f"{path.name}: {len(chunks)} chunk(s)")


In [ ]:
# === 7d. PROMPT: the 7 field definitions and extraction rules ===
# Edit this cell and re-run it if you want to change the prompt, without touching the other cells.

FIELDS = ["baseline", "additionality", "leakage", "permanence_buffer",
          "crediting_period", "area", "estimated_reductions"]

PROMPT = """\
You are extracting facts from part of a carbon project document (Verra VCS).
Read the text below and fill in the 7 fields.
FIELDS:
- baseline: The counterfactual/business-as-usual scenario that plays out if the project does not happen (usually a deforestation rate or baseline emissions in tCO2e)
- additionality: Proof that the project only happens because of carbon finance, and doesn't happen anyway (the test used, e.g. VT0001, investment or barrier analysis, common practice, and its conclusion)
- leakage: Emissions generated from emission-causing activities that would've been merely moved from the project site to another site (a percentage or tCO2e)
- permanence_buffer: A percentage of carbon credits reserved for insurance against project risks
- crediting_period: The period in which the project earns carbon credits, with a start and end date, often measured in years
- area: The geographical scope of the project measured in hectares
- estimated_reductions: The tCO2e avoided, per year or in total, after leakage and the buffer are deducted
RULES:
- Only use facts written in the text below. Never guess or use outside knowledge.
- Copy numbers, units and dates exactly as written (e.g. "<number> hectares", "<start year>–<end year>"). Never copy placeholder examples from these instructions.
- "quote" is the exact sentence the value came from, copied word for word, max 40 words.
- The quote must contain the value itself (the numbers and units). If no sentence supports it, the field is "not stated".
- If a field is not in the text, write only "not stated" for it: no value, page or quote.
- Keep each value short: a number with its unit, a date range, or one sentence.
- additionality is only about whether the project needs carbon finance (e.g. an additionality tool, investment or barrier analysis). Community or biodiversity co-benefits are NOT additionality.
Reply with JSON only, in exactly this shape:
{
  "baseline": {"value": "...", "page": 0, "quote": "..."},
  "additionality": {"value": "...", "page": 0, "quote": "..."},
  "leakage": {"value": "...", "page": 0, "quote": "..."},
  "permanence_buffer": {"value": "...", "page": 0, "quote": "..."},
  "crediting_period": {"value": "...", "page": 0, "quote": "..."},
  "area": {"value": "<number> hectares", "page": 0, "quote": "..."},
  "estimated_reductions": {"value": "<number> tCO2e", "page": 0, "quote": "..."}
}
If a field is not in the text, write "not stated" instead of the object, e.g. "leakage": "not stated"
"""


In [ ]:
# === 7e. EXTRACT: send one chunk to the model and recover the page from the quote ===
# find_page looks up the quote in the chunk text. On bad JSON, retries once, then returns {_error: raw_reply}.

def find_page(chunk, quote):
    if not quote or quote == "not stated":
        return None
    pos = chunk["text"].find(str(quote)[:60])
    if pos == -1:
        return None
    before = re.findall(r"--- page (\d+) ---", chunk["text"][:pos])
    return int(before[-1]) if before else chunk["start_page"]


def run_extraction(chunk, prompt):
    message = (
        f"{prompt}\n\nPages {chunk['start_page']}-{chunk['end_page']}:\n\n"
        f"{chunk['text']}"
    )
    raw = ask(message)
    try:
        result = parse_json(raw)
    except (json.JSONDecodeError, TypeError, ValueError):
        raw = ask(message)
        try:
            result = parse_json(raw)
        except (json.JSONDecodeError, TypeError, ValueError):
            return {"_error": raw}
    if not isinstance(result, dict):
        return {"_error": raw}
    for item in result.values():
        if isinstance(item, dict):
            quote = item.get("quote")
            found = (
                bool(quote)
                and quote != "not stated"
                and chunk["text"].find(str(quote)[:60]) != -1
            )
            item["quote_found"] = found
            item["page"] = find_page(chunk, quote)
    return result


In [ ]:
# === 7f. MERGE: combine per-chunk notes into one record per document ===
# Two filters run before conflicts are decided: the value must appear in its
# own quote (or that sentence in the document), and it must use the right
# units for the field. Failures go under unsupported, never supported.
# If one distinct value survives, it is promoted; identical numbers collapse.

def _stated(item):
    """Turn one field into {value, page, quote}, or None if not stated."""
    if item is None or item == "not stated":
        return None
    if isinstance(item, dict):
        value = item.get("value", "not stated")
        if value is None or value == "not stated" or str(value).strip() == "":
            return None
        return {
            "value": value,
            "page": item.get("page"),
            "quote": item.get("quote"),
            "quote_found": item.get("quote_found"),
        }
    # Sometimes the model returns a bare string instead of the dict.
    if str(item).strip() == "":
        return None
    return {"value": item, "page": None, "quote": None, "quote_found": False}


def _norm_ws(text):
    """Collapse any whitespace (newlines, tabs, double spaces) to one space."""
    return re.sub(r"\s+", " ", str(text)).strip()


def _strip_thousands(text):
    """Drop commas/spaces used as thousands separators, e.g. 169,741 → 169741."""
    return re.sub(r"(?<=\d)[, \u00a0\u202f](?=\d)", "", str(text))


def _extract_numbers(text):
    """Return the numbers in `text`, with commas/spaces already stripped.

    Leading zeros on whole numbers are dropped ("01" → "1") so a date
    written as "01 January 2012" still matches "1 January 2012".
    Decimals such as "30,168.66" are kept as "30168.66".

    Digits that are part of a unit (the "2" in "tCO2e") are ignored,
    otherwise "221,205 tCO2e" would look like the numbers 221205 and 2
    and would not collapse with "221205".
    """
    s = re.sub(r"(?i)\bt?\s*co2e?\b", " ", str(text))
    s = _strip_thousands(s)
    raw = re.findall(r"(?<![A-Za-z])\d+(?:\.\d+)?(?:st|nd|rd|th)?(?![A-Za-z])", s)
    out = []
    for n in raw:
        n = re.sub(r"(?:st|nd|rd|th)$", "", n, flags=re.I)
        if "." in n:
            out.append(n)
        else:
            out.append(str(int(n)))
    return out


def _numbers_in_text(nums, text):
    """True iff every number in `nums` appears as a whole number in `text`.

    "20" will not match inside "2020", but "169741" will match
    "169741.38" (the same area, with extra decimal places).
    """
    hay = _strip_thousands(text)
    for n in nums:
        if not re.search(r"(?<![\d.])" + re.escape(n) + r"(?![\d])", hay):
            return False
    return True


def _locate_quote(quote, doc_text):
    """Find a quote in the document. Return (norm_doc, pos, length) or None.

    Matching ignores case and whitespace, because PDF text often breaks
    a sentence across lines and the model may change capitalisation.
    Very short quotes ("0", "221205") are rejected: they appear all
    over a document and are not reliable evidence on their own.
    """
    if not quote or not doc_text:
        return None
    q = _norm_ws(quote)
    if not q or q.lower() == "not stated":
        return None
    # A real quote is a sentence, or at least a short clause. Anything
    # shorter than 16 characters is usually just the value itself.
    if len(q) < 16:
        return None
    d = _norm_ws(doc_text)
    dl, ql = d.lower(), q.lower()
    pos, q_len = dl.find(ql), len(q)
    if pos == -1:
        for n in (80, 60, 40):
            if len(q) > n:
                pos = dl.find(ql[:n])
                if pos != -1:
                    q_len = n
                    break
    if pos == -1:
        return None
    return d, pos, q_len


def _quote_in_doc(quote, doc_text):
    """True iff the quote appears in the document, ignoring case/whitespace."""
    return _locate_quote(quote, doc_text) is not None


def _page_in_doc(doc_text, quote):
    """Best-effort page number for a quote, using --- page N --- markers."""
    loc = _locate_quote(quote, doc_text)
    if not loc:
        return None
    d, pos, _ = loc
    pages = re.findall(r"--- page (\d+) ---", d[:pos])
    return int(pages[-1]) if pages else None


def _sentence_breaks(d, left, right):
    """Positions of sentence-ending punctuation in d[left:right]."""
    breaks = []
    for i in range(max(0, left), min(len(d), right)):
        if d[i] in ".?!" and (i + 1 >= len(d) or d[i + 1] == " "):
            breaks.append(i)
    return breaks


_ANAPHORA = re.compile(
    r"^\s*(This|These|The above|Applying this|Hence|Therefore|Thus)\b",
    re.I,
)


def _evidence_span(doc_text, quote):
    """The document sentence that contains the quote, or the quote itself.

    The prompt caps quotes at 40 words, so the number is often cut off
    ("...Kasigau Corridor REDD P" instead of "...total 169,741 hectares").
    We therefore read the rest of that same sentence, but not the
    sentences around it — those often contain other figures.

    Exception: if the sentence starts with a back-reference such as
    "Applying this factor...", we also include the previous sentence,
    because that is usually where the number sits.
    """
    if not quote:
        return ""
    q = _norm_ws(quote)
    loc = _locate_quote(quote, doc_text) if doc_text else None
    if not loc:
        return q
    d, pos, q_len = loc
    # Start at the quote itself. A short look-back (one clause, not a
    # whole form or table) is allowed if the model skipped a few words
    # at the start of the sentence.
    breaks_before = _sentence_breaks(d, max(0, pos - 120), pos)
    start = breaks_before[-1] + 1 if breaks_before else pos
    # If the quote already ends at a sentence boundary, stop there.
    # Otherwise look forward just far enough to finish a truncated
    # 40-word quote, and stop at the first sentence break.
    matched = d[pos:pos + q_len].rstrip()
    if matched[-1:] in ".?!":
        end = pos + q_len
    else:
        cap = min(len(d), pos + q_len + 280)
        breaks_after = _sentence_breaks(d, pos + q_len, cap)
        end = breaks_after[0] + 1 if breaks_after else cap
    sentence = d[start:end].strip()
    if _ANAPHORA.search(sentence):
        prev_breaks = _sentence_breaks(d, max(0, start - 400), start)
        if prev_breaks:
            sentence = d[prev_breaks[-1] + 1:end].strip()
        else:
            sentence = d[max(0, start - 400):end].strip()
    return sentence


_ZERO_WORDS = re.compile(
    r"\b(?:zero|none|nil)\b|"
    r"no displacement|no leakage|"
    r"leakage (?:emissions )?are accounted as zero|"
    r"leakage is considered as (?:0|zero)|"
    r"accounted as zero",
    re.I,
)


def _value_in_quote(value, evidence):
    """True iff `value` is actually supported by the quote/sentence.

    For values that contain numbers (including a period "2011 to 2041"),
    every number must appear in the evidence. "0 tCO2e" is also accepted
    when the sentence says "zero" rather than the digit 0.
    For descriptive values with no digits, a substantial piece of the
    wording must appear.
    """
    if not evidence:
        return False
    ev = _norm_ws(evidence)
    val = _norm_ws(value)
    if not val:
        return False
    nums = _extract_numbers(val)
    if nums:
        # For a period such as "2005-01-01 to 2034-12-31" or
        # "14 March 2011 – 13 March 2041", the years are what matter.
        # Day and month numbers often appear as words ("December") in
        # the sentence, so we do not require those.
        years = [n for n in nums if re.fullmatch(r"\d{4}", n)]
        if years:
            return _numbers_in_text(years, ev)
        if _numbers_in_text(nums, ev):
            return True
        # "emissions from leakage are zero" supports a value of 0.
        if nums == ["0"] and _ZERO_WORDS.search(ev):
            return True
        return False
    ev_l, val_l = ev.lower(), val.lower()
    if val_l in ev_l:
        return True
    if len(val_l) >= 40 and val_l[:40] in ev_l:
        return True
    words = [w for w in re.findall(r"[a-z]{4,}", val_l)]
    if not words:
        return val_l in ev_l
    hits = sum(1 for w in words if re.search(r"\b" + re.escape(w) + r"\b", ev_l))
    return hits >= max(3, int(0.6 * len(words)))


# What a valid value is allowed to look like, by field.
# Kept simple on purpose: we only reject obvious mismatches
# (a hectare figure filed under baseline, a % filed as area, etc.).
_AREA_UNIT = re.compile(r"\b(ha|hectares?|acres?)\b", re.I)
_PERCENT = re.compile(r"%|\bpercent(?:age)?\b", re.I)
_TCO2 = re.compile(
    r"\bt\s*co2e?\b|\btco2e\b|\btonnes?\b|\btons?\b|\bm\.?t\.?\b|\bghg\b",
    re.I,
)
_PERIOD = re.compile(
    r"\b(?:years?|year|months?|days?|"
    r"january|february|march|april|may|june|july|august|"
    r"september|october|november|december|"
    r"jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec)\b|"
    r"\d{4}\s*[-–—/to]+\s*\d{4}|"
    r"\b\d{4}\b|"
    r"\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}",
    re.I,
)
_RATE = re.compile(r"/year|\bper year\b|\bper yr\b|\brate\b|\bdeforestation\b", re.I)
_BARE_PERCENT = re.compile(r"^[\d,.\s]+\s*(%|percent|percentage)\s*$", re.I)
_BARE_AREA = re.compile(
    r"^[\d,.\s]+\s*(ha|hectares?|acres?)\s*$", re.I
)


def _field_units_ok(field, value, evidence=""):
    """True iff `value` looks like the right kind of thing for `field`.

    Units may sit on the value or in the supporting sentence, because
    the model sometimes drops them ("3,871" with a quote that says
    "3,871 ha").
    """
    v = str(value)
    blob = f"{v} {evidence}"
    if field == "area":
        return bool(_extract_numbers(v) and _AREA_UNIT.search(blob))
    if field in ("permanence_buffer", "leakage"):
        # A percentage, a tCO2e / tonne deduction, or an explicit zero.
        if re.search(r"\bzero\b", v, re.I) or _extract_numbers(v) == ["0"]:
            return True
        if _extract_numbers(v) and (_PERCENT.search(blob) or _TCO2.search(blob)):
            return True
        return False
    if field == "estimated_reductions":
        return bool(_extract_numbers(v) and _TCO2.search(blob))
    if field == "crediting_period":
        return bool(_PERIOD.search(v))
    if field in ("baseline", "additionality"):
        # These are descriptive. Reject a bare area ("169,741 hectares"),
        # a bare percentage ("39.90%"), or a bare number with no words
        # ("55,244.75", often a table parameter). A rate such as
        # "10,738.9 hectares/year" is allowed for baseline.
        if re.fullmatch(r"[+\-]?[\d,.\s]+", v.strip()):
            return False
        if re.search(r"value applied\s*:", evidence, re.I):
            return False
        if _BARE_AREA.search(v.strip()) and not _RATE.search(v):
            return False
        if _BARE_PERCENT.search(v.strip()):
            return False
        # A bare tCO2e figure is only a baseline if the evidence
        # actually talks about baseline emissions, not project benefits.
        if field == "baseline" and _TCO2.search(v) and not re.search(r"baseline", blob, re.I):
            return False
        return True
    return True


def _same_value(a, b):
    """True iff two values are the same number (or the same wording).

    "3,871", "3,871 ha" and "3,871 hectares" collapse together.
    "01 January 2010 – 31 December 2039" and "2010-2039" collapse
    because they share the same year endpoints. "30 years" and
    "2016-2045" do not, because one has no years.
    """
    if _norm_ws(a).lower() == _norm_ws(b).lower():
        return True
    na, nb = _extract_numbers(a), _extract_numbers(b)
    if not na or not nb:
        return False
    years_a = tuple(n for n in na if re.fullmatch(r"\d{4}", n))
    years_b = tuple(n for n in nb if re.fullmatch(r"\d{4}", n))
    if years_a and years_b:
        return years_a == years_b
    return na == nb


def _collapse(items):
    """Group candidates that are the same number; keep the richest wording."""
    groups = []
    for item in items:
        placed = False
        for group in groups:
            if _same_value(group[0]["value"], item["value"]):
                group.append(item)
                placed = True
                break
        if not placed:
            groups.append([item])
    collapsed = []
    for group in groups:
        keep = dict(max(
            group,
            key=lambda x: (
                len(str(x.get("value") or "")),
                len(str(x.get("quote") or "")),
            ),
        ))
        pages = []
        for x in group:
            if x.get("page") is not None and x["page"] not in pages:
                pages.append(x["page"])
            for p in x.get("pages") or []:
                if p not in pages:
                    pages.append(p)
        keep["pages"] = pages
        collapsed.append(keep)
    return collapsed


def _classify(item, field, doc_text=None):
    """Return (bucket, reason, item) where bucket is supported|unsupported."""
    item = dict(item)
    quote = item.get("quote")
    value = item.get("value")

    if doc_text:
        quote_ok = _quote_in_doc(quote, doc_text)
        if quote_ok and item.get("page") is None:
            item["page"] = _page_in_doc(doc_text, quote)
        item["quote_found"] = quote_ok
    else:
        # No document text: fall back to the flag set at extraction time.
        quote_ok = item.get("quote_found") is True

    if not quote_ok:
        return "unsupported", "quote not in document", item

    evidence = _evidence_span(doc_text, quote) if doc_text else _norm_ws(quote)
    if not _value_in_quote(value, evidence):
        return "unsupported", "value not in quote", item
    if not _field_units_ok(field, value, evidence):
        return "unsupported", "wrong unit or type for field", item
    return "supported", None, item


def merge_notes(chunk_results, doc_text=None):
    """Merge a list of per-chunk dicts into one record for the document.

    Pass `doc_text` (the extracted PDF text) whenever you have it.
    That lets us re-check quotes across line breaks and recover a
    page number for candidates the original quote_found flag missed.
    """
    merged = {}
    for field in FIELDS:
        found, unsupported = [], []
        for result in chunk_results:
            if not isinstance(result, dict) or "_error" in result:
                continue
            item = _stated(result.get(field))
            if not item:
                continue
            bucket, reason, item = _classify(item, field, doc_text)
            if bucket == "supported":
                found.append(item)
            else:
                item["reject"] = reason
                unsupported.append(item)

        # Filters run before any conflict decision.
        found = _collapse(found)

        if not found:
            merged[field] = (
                {"value": "not stated", "unsupported": unsupported}
                if unsupported else "not stated"
            )
            continue

        if len(found) == 1:
            keep = dict(found[0])
            if unsupported:
                keep["unsupported"] = unsupported
            merged[field] = keep
        else:
            rec = {"value": "conflict", "conflicts": found}
            if unsupported:
                rec["unsupported"] = unsupported
            merged[field] = rec

    return merged


In [ ]:
# === 7g. TEST: run extraction on the Kasigau-II PD excerpt ===
# One short file, so you can inspect the merged JSON before the full run.

import json

name = "612_Kasigau-II-Community-Ranches_PD_excerpt.txt"
notes = [run_extraction(c, PROMPT) for c in all_chunks[name]]
doc_text = (Path(TALLY) / "text" / name).read_text(encoding="utf-8")
print(json.dumps(merge_notes(notes, doc_text=doc_text), indent=2, ensure_ascii=False))


In [ ]:
# === 7h. FULL RUN: extract every document, saving results as you go ===
# Skips docs that already have a result file. Prints progress and a summary table at the end.

import time
from pathlib import Path

results_dir = Path(TALLY) / "results"
results_dir.mkdir(parents=True, exist_ok=True)

torch.cuda.empty_cache()
summary = []
for name, chunks in all_chunks.items():
    out_path = results_dir / f"{Path(name).stem}.json"
    if out_path.exists():
        print(f"skip {name} (already have {out_path.name})")
        data = json.loads(out_path.read_text(encoding="utf-8"))
        n_err = sum(
            1 for n in data.get("chunks", [])
            if isinstance(n, dict) and "_error" in n
        )
        summary.append({
            "doc": name,
            "chunks": len(chunks),
            "status": "skipped",
            "errors": n_err,
            "seconds": data.get("elapsed_s"),
        })
        continue

    notes = []
    t_doc = time.time()
    for i, chunk in enumerate(chunks, start=1):
        torch.cuda.empty_cache()
        t0 = time.time()
        note = run_extraction(chunk, PROMPT)
        elapsed = time.time() - t0
        notes.append(note)
        flag = " ERROR" if isinstance(note, dict) and "_error" in note else ""
        print(f"{name}  chunk {i}/{len(chunks)}  {elapsed:.1f}s{flag}")

    payload = {
        "doc": name,
        "n_chunks": len(chunks),
        "elapsed_s": round(time.time() - t_doc, 1),
        "chunks": notes,
        "merged": merge_notes(notes, doc_text=(Path(TALLY) / "text" / name).read_text(encoding="utf-8")),
    }
    out_path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    n_err = sum(1 for n in notes if isinstance(n, dict) and "_error" in n)
    summary.append({
        "doc": name,
        "chunks": len(chunks),
        "status": "done",
        "errors": n_err,
        "seconds": payload["elapsed_s"],
    })
    print(f"saved {out_path}")

print("\n=== summary ===")
print(f"{'document':<60} {'chunks':>6} {'err':>4} {'sec':>8}  status")
for row in summary:
    sec = row["seconds"] if row["seconds"] is not None else "-"
    print(f"{row['doc']:<60} {row['chunks']:>6} {row['errors']:>4} {sec:>8}  {row['status']}")
